# Restriction-map figures for the paper

A tidy, self-contained subset of `restriction_map_mlp_analysis.ipynb`, ported for publication
figures. The setup cells (§0 imports, §1 harvest) are **copied verbatim** from that notebook.

**Figures**
1. **SVD reshaping** — what each restriction map does to an LSTM vector: the singular spectrum
   (stretch, raw magnitudes) and the polar rotation vs stretch split, both vs the true
   initial maps (untrained sheaf-learner on the trained embeddings).
2. **Sheaf diffusion operator over nodes** — one diffusion step decomposed on the scalp.
3. **Per-window ‖ρ‖ snapshots** — six *adjacent* windows in time.
4. **Embedding spread** — per-edge map clouds, comparing the base vs LapPE run.
5. **Relative update per node** — ‖Δh‖/‖h‖ on the scalp (azimuthal projection).

**Toggles** (top of the helpers cell): `PROJECTION` (`regular` default | `azimuthal`),
`SHOW_SPATIAL_EDGES` + `N_SPATIAL_EDGES` (a faint electrode-proximity scaffold behind the graph).
The scalp figures iterate over **every** loaded EEG run.

## §0 · Config + imports

In [ ]:
import os, sys, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(42)

PROJECT_ROOT = os.environ.get(
    "BRAINDYN_ROOT", "~/project_<account>/<netid>/brainy/BrainDyn-SUMRY"
)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
import analysis.restriction_maps as rm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

# ── Checkpoints + data (override via env) ─────────────────────────────────────
# Each checkpoint MUST be an MLP-map run (sheaf_mlp_maps=True).
DATASETS = {
    #"EEG": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/paper_eeg/braindyn_eeg_short_granger_fold4.pt"),
     #            manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
    "EEG_best": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/paper_eeg/braindyn_eeg_short_pe_fold4.pt"),
                 manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
}

DATASETS = {k: v for k, v in DATASETS.items() if v["ckpt"]}

MAX_BATCHES = int(os.environ.get("MAX_BATCHES", "4"))   # probe batches to pool
BATCH_SIZE  = int(os.environ.get("BATCH_SIZE", "64"))
SUBSAMPLE   = int(os.environ.get("SUBSAMPLE", "1500"))  # cap for O(P^2) / Jacobian ops
CONDITION   = os.environ.get("CONDITION", "EC")

print("configured datasets:", list(DATASETS) or "(none — set FMRI_CKPT / EEG_CKPT)")
print("device:", device)

# Optional extra checkpoints to compare as "conditions" (same node space; e.g. EC vs EO)
# for the cross-condition Gram in §4c. COND_CKPTS is comma-separated; COND_MANIFESTS pairs
# with it (or a single manifest reused); COND_KIND is fmri|eeg.
_cc = [p for p in os.environ.get("COND_CKPTS", "").split(",") if p]
_cm = [p for p in os.environ.get("COND_MANIFESTS", "").split(",") if p]
_cn = [p for p in os.environ.get("COND_NAMES", "").split(",") if p]
COND_KIND = os.environ.get("COND_KIND", "eeg")
CONDITIONS = [dict(name=(_cn[i] if i < len(_cn) else f"cond{i+1}"), ckpt=ck, kind=COND_KIND,
                   manifest=(_cm[i] if i < len(_cm) else (_cm[0] if _cm else "")))
              for i, ck in enumerate(_cc)]
print("conditions (cross-condition Gram):", [c["name"] for c in CONDITIONS] or "(none)")

## §1 · Load checkpoints & harvest maps

In [ ]:
from data.rbc_dataset import make_dataloaders as make_fmri_loaders
from data.lemon_dataset import make_lemon_dataloaders


def build_loaders(dc, cfg):
    """Build the full split dict of DataLoaders for the dataset."""
    g = cfg.get
    x, y, stride = g("x", 30), g("y", 10), g("stride", 10)
    if dc["kind"] == "fmri":
        return make_fmri_loaders(
            dc["manifest"], x=x, y=y, stride=stride, batch_size=BATCH_SIZE,
            num_workers=0, pin_memory=False, norm_mode=g("norm_mode", "train_global"),
        )
    return make_lemon_dataloaders(
        manifest_csv=dc["manifest"], x=x, y=y, stride=stride, batch_size=BATCH_SIZE,
        num_workers=0, condition=g("condition", CONDITION),
        max_subjects=g("max_subjects"), max_windows_per_subject=g("max_windows_per_subject"),
        seed=g("lemon_data_seed", 42), pin_memory=False, norm_mode=g("norm_mode", "train_global"),
    )


def pick_split(loaders, prefer=("test", "val", "train")):
    """Return (loader, split_name) for the first available split."""
    for split in prefer:
        if loaders.get(split) is not None and len(loaders[split].dataset) > 0:
            return loaders[split], split
    raise RuntimeError("no usable split in loaders")


HARVEST = {}
for name, dc in DATASETS.items():
    ckpt = torch.load(dc["ckpt"], map_location="cpu", weights_only=False)
    ei = ckpt["edge_index"].to(device)
    N = rm.num_nodes_from_ckpt(ckpt)
    model = rm.build_model_from_ckpt(ckpt, ei, device=device, num_nodes=N, ckpt_name=dc["ckpt"])
    loaders = build_loaders(dc, ckpt["config"])
    loader, split = pick_split(loaders)
    H = rm.harvest_over_loader(model, loader, ei, norm_stats=ckpt.get("norm_stats"),
                               max_batches=MAX_BATCHES, device=device)
    # True initial map field: the untrained sheaf-learner (identity or gain=0.1
    # readout, per the run's identity_restriction_init) on the *same* trained
    # embeddings. Replaces the old Xavier-random "init" null in Figure 1.
    H_init = rm.harvest_initial_baseline(model, loader, ei, norm_stats=ckpt.get("norm_stats"),
                                         max_batches=MAX_BATCHES, device=device)
    HARVEST[name] = dict(H=H, H_init=H_init, model=model, ckpt=ckpt, ei=ei, N=N, E=int(ei.shape[1]),
                         loaders=loaders, split=split,
                         dt=float(ckpt["config"].get("dt", 1.0) or 1.0))
    R = H["rho_src"]
    print(f"{name:5s}  N={N:4d}  E={ei.shape[1]:5d}  maps {R.shape}  split={split}  "
          f"(H={R.shape[-2]}, d_e={R.shape[-1]}, square={R.shape[-2] == R.shape[-1]})")
assert HARVEST, "No datasets harvested — set FMRI_CKPT / EEG_CKPT."
n = len(HARVEST)                       # reused for subplot layouts in later sections
summary = {name: {} for name in HARVEST}   # per-dataset scalars, filled across sections

## Helpers — projection toggle, spatial scaffold, scalp drawing

`PROJECTION` picks the scalp layout (**regular** top-down drop-z by default, or **azimuthal**
MNE-style). `SHOW_SPATIAL_EDGES` draws the `N_SPATIAL_EDGES` shortest electrode–electrode links in
faint grey behind the graph so a sparse Granger graph still reads as a brain. `EEG_NAMES` is every
EEG run that harvested — the scalp figures loop over all of them.

In [ ]:
from matplotlib.collections import LineCollection
from pathlib import Path as _Path
from data.lemon_dataset import montage_positions

# ── toggles ──────────────────────────────────────────────────────────────────
PROJECTION         = "regular"     # "regular" (top-down drop-z, default) | "azimuthal" (MNE topomap)
SHOW_SPATIAL_EDGES = True          # faint grey electrode-proximity scaffold behind the graph
N_SPATIAL_EDGES    = 400            # number of nearest-electrode links to draw

# professional defaults for the exported figures
plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 220, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

# every EEG checkpoint that harvested — the scalp figures iterate over all of them
EEG_NAMES = [k for k, v in DATASETS.items() if v["kind"] == "eeg" and k in HARVEST]
print("EEG runs for the scalp figures:", EEG_NAMES or "(none)")

def scalp_xy(pos3d, mode=None):
    # mode overrides the global PROJECTION toggle for a single figure.
    p = np.asarray(pos3d, float)
    if (mode or PROJECTION) == "regular":
        return p[:, :2].copy()                              # top-down (x = right, y = nose)
    q = p - np.array([0.0, 0.0, p[:, 2].min()])             # azimuthal-equidistant (nose up)
    r = np.linalg.norm(q, axis=1); r[r == 0] = 1.0
    theta = np.arccos(np.clip(q[:, 2] / r, -1.0, 1.0))      # 0 at vertex -> outward
    phi = np.arctan2(q[:, 1], q[:, 0])                      # +y = nose, +x = right
    return np.c_[theta * np.cos(phi), theta * np.sin(phi)]

def montage(rec, mode=None):
    # (positions3d (N,3), pos2d (N,2)) for a harvested record, honoring the projection toggle.
    ds = rec["loaders"]["train"].dataset
    ch = list(ds.channels)
    pj = _Path(ds._samples[0][0]).parent / "montage_positions.json"
    pos3d = montage_positions(ch, positions_json=str(pj) if pj.exists() else None)
    return pos3d, scalp_xy(pos3d, mode)

def spatial_scaffold(pos3d, n_edges):
    # the n_edges shortest electrode-electrode links (undirected), as node-index pairs.
    if not n_edges:
        return []
    D = np.linalg.norm(pos3d[:, None, :] - pos3d[None, :, :], axis=-1)
    iu, ju = np.triu_indices(len(pos3d), 1)
    order = np.argsort(D[iu, ju])[:int(n_edges)]
    return list(zip(iu[order].tolist(), ju[order].tolist()))

def _head(ax, pos2d):
    r = np.abs(pos2d).max() * 1.15; th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), color="k", lw=1)                            # head
    ax.plot([-0.09 * r, 0, 0.09 * r], [r * 0.99, r * 1.12, r * 0.99], color="k", lw=1)  # nose

def _scaffold(ax, pos2d, pairs):
    for i, j in pairs:
        ax.plot([pos2d[i, 0], pos2d[j, 0]], [pos2d[i, 1], pos2d[j, 1]],
                color="#e6e6e6", lw=0.8, zorder=0)

def draw_topo_edges(ax, ei_np, pos2d, edge_vals, title, scale01=True, cmap="viridis",
                    vmin=None, vmax=None, spatial=None):
    _head(ax, pos2d)
    if spatial:
        _scaffold(ax, pos2d, spatial)
    v = np.asarray(edge_vals, float)
    if scale01:
        lo, hi = np.nanmin(v), np.nanmax(v); v = (v - lo) / (hi - lo + 1e-12); vmin, vmax = 0.0, 1.0
    else:
        vmin = 0.0 if vmin is None else vmin
        vmax = np.nanmax(v) if vmax is None else vmax
    segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])] for s, d in zip(ei_np[0], ei_np[1])]
    lc = LineCollection(segs, cmap=cmap, norm=plt.Normalize(vmin, vmax), lw=1.6, alpha=0.85, zorder=2)
    lc.set_array(v); ax.add_collection(lc)
    ax.scatter(pos2d[:, 0], pos2d[:, 1], s=18, c="k", zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return lc

def draw_topo_nodes(ax, ei_np, pos2d, node_vals, title, cmap="magma", vmin=None, vmax=None, spatial=None):
    _head(ax, pos2d)
    if spatial:
        _scaffold(ax, pos2d, spatial)
    segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])] for s, d in zip(ei_np[0], ei_np[1])]
    ax.add_collection(LineCollection(segs, colors="#cfcfcf", lw=0.5, alpha=0.7, zorder=1))
    v = np.asarray(node_vals, float)
    sc = ax.scatter(pos2d[:, 0], pos2d[:, 1], s=70, c=v, cmap=cmap,
                    vmin=(np.nanmin(v) if vmin is None else vmin),
                    vmax=(np.nanmax(v) if vmax is None else vmax),
                    edgecolors="k", linewidths=0.4, zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return sc

## Figure 1 · Normalized SVD reshaping — stretching vs scaling

Left panel toggles via `NORMALIZE_SPECTRUM`: `σ₁`-normalized (shape, gauge-invariant under
`sheaf_norm=sym`) or **raw `σₖ`** to check magnitude growth from init. Shown with a
the true initial maps as the null (untrained sheaf-learner on the trained
embeddings, per `identity_restriction_init`), plus the polar split into anisotropic **stretch** vs **rotation**.

In [ ]:
# Figure 1 · SVD reshaping + rotation-vs-stretch. NORMALIZE_SPECTRUM toggles the left panel:
#   True  -> sigma_1-normalized spectrum (shape only; a gauge under sheaf_norm=sym).
#   False -> raw sigma_k, so you can see whether the spectrum *grows* in magnitude from
#            initialization (identity/gain=0.1) to the learned maps.
# The Initial curve is the untrained sheaf-learner on the trained embeddings
# (rm.harvest_initial_baseline).
NORMALIZE_SPECTRUM = False
C_LEARN, C_XAV = "#1f77b4", "#9aa0a6"
nrows = len(HARVEST)
fig, axes = plt.subplots(nrows, 2, figsize=(12.5, 4.4 * nrows), squeeze=False)
for row, (name, rec) in enumerate(HARVEST.items()):
    R = np.asarray(rec["H"]["rho_src"], float); H, De = R.shape[-2], R.shape[-1]
    R2 = R.reshape(-1, H, De)
    # Real initial condition: our maps come from an MLP on LSTM embeddings, so
    # "init" is that untrained MLP's output on the same embeddings (identity or
    # small gain=0.1 readout) — NOT a Xavier-random matrix population.
    init = np.asarray(rec["H_init"]["rho_src"], float).reshape(-1, H, De)
    sv, isv = rm.svdvals(R2), rm.svdvals(init)
    er, ier = rm.effective_rank(R2).mean(), rm.effective_rank(init).mean()
    k = np.arange(1, sv.shape[1] + 1)
    if NORMALIZE_SPECTRUM:
        svp = sv / (sv[:, :1] + 1e-12); isvp = isv / (isv[:, :1] + 1e-12)
        ylab, kind = r"$\sigma_k/\sigma_1$", "normalized"
    else:
        svp, isvp = sv, isv
        ylab, kind = r"$\sigma_k$", "raw"

    a0 = axes[row, 0]
    a0.fill_between(k, *np.percentile(svp, [25, 75], 0), color=C_LEARN, alpha=0.18)
    a0.plot(k, svp.mean(0), "-o", ms=4, lw=2, color=C_LEARN, label="Learned")
    a0.fill_between(k, *np.percentile(isvp, [25, 75], 0), color=C_XAV, alpha=0.18)
    a0.plot(k, isvp.mean(0), "--s", ms=3, lw=1.6, color=C_XAV, label="Initial")
    a0.set_xlabel("singular index  k"); a0.set_ylabel(ylab)
    a0.set_ylim(0, 1.02) if NORMALIZE_SPECTRUM else a0.set_ylim(bottom=0)
    a0.set_title(f"{name} — {kind} spectrum   eff.rank {er:.2f} vs {ier:.2f}"); a0.legend()

    oc, spd = rm.orthogonal_content(R2), rm.svd_polar(R2)
    ocx, spdx = rm.orthogonal_content(init), rm.svd_polar(init)
    a1 = axes[row, 1]; xx = np.arange(2); w = 0.36
    L = [oc["anisotropy"].mean(), spd["rotation_frac"].mean()]
    Le = [oc["anisotropy"].std(), spd["rotation_frac"].std()]
    X = [ocx["anisotropy"].mean(), spdx["rotation_frac"].mean()]
    Xe = [ocx["anisotropy"].std(), spdx["rotation_frac"].std()]
    a1.bar(xx - w / 2, L, w, yerr=Le, capsize=4, color=C_LEARN, label="learned")
    a1.bar(xx + w / 2, X, w, yerr=Xe, capsize=4, color=C_XAV, label="Initial")
    a1.set_xticks(xx); a1.set_xticklabels(["anisotropic\nstretch", "rotation\nfraction"])
    a1.set_ylim(0, 1); a1.set_ylabel("fraction")
    a1.set_title(f"{name} — stretching vs scaling"); a1.legend()
_spec = "normalized" if NORMALIZE_SPECTRUM else "raw"
fig.suptitle(f"Learned maps: {_spec} spectrum + rotation vs stretch", fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.98]); plt.show()


### Prep · one-step sheaf decomposition (feeds Figure 3) + per-edge diffusion gain (feeds Figure 2)

In [ ]:
STEP_DECOMP = {}
for name in EEG_NAMES:
    rec = HARVEST[name]; loader, _ = pick_split(rec["loaders"])
    STEP_DECOMP[name] = rm.sheaf_step_decomposition_over_loader(
        rec["model"], loader, rec["ei"], norm_stats=rec["ckpt"].get("norm_stats"),
        max_batches=MAX_BATCHES, device=device, round_idx=0)
print("step decomposition computed for:", list(STEP_DECOMP) or "(no EEG)")

In [ ]:
# Prep · per-(window, edge) diffusion gain for Figure 2 (eff_step*||msg|| each half-edge delivers, with
# the learned diffusion_gain folded in). Absolute by default; GAIN_MODE=relative divides by receiving ||h||.
import torch
from main import batch_to_model_tensors
GAIN_MODE = os.environ.get("GAIN_MODE", "absolute")

def harvest_diffusion_gain(rec, gain_mode=GAIN_MODE):
    model, ei = rec["model"], rec["ei"]
    gl = model.dynamics.graph_laplacian
    g = float(gl.diffusion_gain()) if hasattr(gl, "diffusion_gain") else 1.0
    src = ei[0].detach().cpu().numpy(); dst = ei[1].detach().cpu().numpy()
    loader, _ = pick_split(rec["loaders"])
    cs_l, cd_l, hs_l, hd_l = [], [], [], []
    for i, batch in enumerate(loader):
        if i >= MAX_BATCHES:
            break
        x, _ = batch_to_model_tensors(batch, device, norm_stats=rec["ckpt"].get("norm_stats"))
        ec = rm.sheaf_edge_contributions(model, x, ei, round_idx=0)
        cs_l.append((g * ec["c_to_src"]).detach().cpu().numpy())
        cd_l.append((g * ec["c_to_dst"]).detach().cpu().numpy())
        if gain_mode == "relative":
            sd = rm.sheaf_step_decomposition(model, x, ei, round_idx=0)
            hn = sd["h_norm"].detach().cpu().numpy()
            hs_l.append(hn[:, src]); hd_l.append(hn[:, dst])
    cs = np.concatenate(cs_l, 0); cd = np.concatenate(cd_l, 0)
    # Fold in the OUTPUT degree normalization (D^{-1/2} at the receiving node) that forward applies
    # after aggregation but sheaf_edge_contributions omits (ones for sheaf_norm="none").
    sc = gl._node_scale(ei, ei.device, torch.float32).detach().cpu().numpy()
    cs = cs * sc[src]; cd = cd * sc[dst]
    if gain_mode == "relative":
        cs = cs / (np.concatenate(hs_l, 0) + 1e-12); cd = cd / (np.concatenate(hd_l, 0) + 1e-12)
    return dict(c_to_src=cs, c_to_dst=cd, src=src, dst=dst, gain=g, mode=gain_mode)

DIFF_GAIN = {name: harvest_diffusion_gain(HARVEST[name]) for name in EEG_NAMES}
print("diffusion gain harvested for:", list(DIFF_GAIN) or "(no EEG)", "| GAIN_MODE:", GAIN_MODE)

## Figure 2 · Per-window diffusion gain — six adjacent windows

The same adjacent-window scalp snapshots as the old `‖ρ‖` figure, but coloured by the **diffusion
gain** each edge actually delivers (`eff_step·‖msg‖`, learned `diffusion_gain` folded in).

In [ ]:
# Figure 2 · Per-window diffusion gain — six ADJACENT windows on the scalp (replaces the ||rho|| snapshots).
# Each edge coloured by eff_step*||msg|| averaged across its two halves; shared colour scale across windows.
SNAPSHOT_START = 70  # None = center the 6-window run; else an int start index
for name in EEG_NAMES:
    rec = HARVEST[name]; dg = DIFF_GAIN[name]
    edge_win = 0.5 * (dg["c_to_src"] + dg["c_to_dst"])              # (S, E)
    ei_np = rec["ei"].detach().cpu().numpy()
    pos3d, pos2d = montage(rec, "regular")
    sp = spatial_scaffold(pos3d, N_SPATIAL_EDGES) if SHOW_SPATIAL_EDGES else []
    S = edge_win.shape[0]; kk = min(6, S)
    start = (max(0, S // 2 - kk // 2) if SNAPSHOT_START is None else int(SNAPSHOT_START))
    start = min(start, max(0, S - kk)); snap = np.arange(start, start + kk)
    gmax = float(np.nanpercentile(edge_win[snap], 99))
    fig, axes = plt.subplots(2, 3, figsize=(15, 9.6), dpi=300); lc = None
    for ax, w in zip(axes.ravel(), snap):
        lc = draw_topo_edges(ax, ei_np, pos2d, edge_win[w], f"window {w}", scale01=False,
                             vmin=0.0, vmax=gmax, spatial=sp)
    for ax in axes.ravel()[len(snap):]:
        ax.axis("off")
    if lc is not None:
        fig.colorbar(lc, ax=axes, fraction=0.02, label=f"diffusion gain (shared scale, {dg['mode']})")
    fig.suptitle(f"{name}: per-window diffusion gain — 6 adjacent windows "
                 f"(t={snap[0]}…{snap[-1]}, g={dg['gain']:.2g})", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 0.9, 0.96]); plt.show()
if not EEG_NAMES:
    print("No EEG dataset configured — skipping diffusion-gain snapshots.")

## Figure 3 · Relative update per node — neighbour-driven state change, DMN-style views

**Read this before trusting the raw number.** One diffusion round updates each node by
$\Delta h = -\,\mathrm{eff\_step}\cdot L_F h$, and the sheaf Laplacian splits into a **diagonal
(self) block** and an **off-diagonal (neighbour) block**:

$$L_F h_i \;=\; \underbrace{\Big(\textstyle\sum_{e\ni i}\rho_{i\to e}^{\top}\rho_{i\to e}\Big)h_i}_{\text{self } D_F\,h}\;-\;\underbrace{\textstyle\sum_{e:(i,j)}\rho_{i\to e}^{\top}\rho_{j\to e}\,h_j}_{\text{neighbour } W_F\,h}.$$

The self block $D_F$ just **re-scales a node's own stalk**, and its size grows with
**node degree $\times\;\lVert\rho\rVert^2$**. So the naive $\lVert\Delta h\rVert/\lVert h\rVert$ is
dominated by graph degree — it lights up the vertex/central hubs (many Granger edges) and goes dark
at the rim. That is a *degree map*, not a *dynamics map*, which is why the values looked
implausibly large (tens ×).

**Toggles** (top of the cell, all env-overridable):

- **`REL_MODE`** — what to plot: `message` (default) $\lVert\Delta h_{\text{msg}}\rVert/\lVert h\rVert$,
  the neighbour (off-diagonal) part only; `reconfig`
  $\lVert\Delta h_{\text{msg}}\rVert/\lVert\Delta h_{\text{self}}\rVert$ (degree-free: $>1$ = the
  graph rewrites the node); `full` the original $\lVert\Delta h\rVert/\lVert h\rVert$. The cell
  prints all three medians + `sheaf_norm`/`diffusion_step`/gain so the magnitude source is explicit.
- **`REL_VIEWS`** — comma list of `sagittal` / `axial` / `posterior`. Default `"sagittal,axial"`
  (two panels, echoing the DMN figure); set `"axial"` for a single top-down topomap.
- **`REL_STYLE`** — `smooth` (Gaussian-kernel field, fMRI-activation look) or `nodes` (discrete
  electrode dots + Granger edges).
- **`REL_CMAP`** — any colormap (`inferno` default, or `viridis`/`magma`/…).
- **`REL_SMOOTH`** — smooth bandwidth as a fraction of head radius (default `0.25`); lower = tighter
  peaks, higher = broader blur. Ignored for `nodes` style.

*Honest caveat:* the field is a smooth posterior-vs-hub gradient, not discrete sub-networks — the
two-view pair makes the gradient legible, but the data don't support a clean DMN-style subnetwork
parcellation on the scalp. Also, scalp electrodes sit on the *upper* skull, so the sagittal/posterior
panels render as an upper-head dome (sensors don't wrap underneath).

In [ ]:
# Figure 3 · Relative update per node — neighbour-driven, DMN-style scalp views.
# WHY the magnitude is ~tens (not ~1):  Δh = −eff_step·L_F·h and L_F = D_F − W_F. The diagonal
# self-block D_F,i = Σ_e ρ_{i→e}ᵀρ_{i→e} scales as (degree × ‖ρ‖²).  This run's sheaf_norm="sym"
# divides L by scalar node DEGREE (edge count) only — it cancels the number of edges but NOT the
# learned map magnitude.  These maps trained large (‖ρ‖_F ≈ 20 vs 4 for identity → ρᵀρ eigenvalues
# ≈ 20–50), so ||Δh||/||h|| sits near the mean ρᵀρ eigenvalue.  The scale is a learned GAUGE (ρ can
# be rescaled without changing predictions after the readout), so the SPATIAL PATTERN is the result;
# the absolute colorbar number is not.  Use REL_MODE="reconfig" (off/diag, ~O(1), gauge-invariant)
# for a scale-free headline.
# ── toggles (all env-overridable) ─────────────────────────────────────────────
REL_MODE   = os.environ.get("REL_MODE", "message")        # message | reconfig | full  (what to plot)
REL_VIEWS  = os.environ.get("REL_VIEWS", "sagittal,axial")# comma list; "axial" = single default view
REL_STYLE  = os.environ.get("REL_STYLE", "smooth")        # "smooth" (RBF field) | "nodes" (discrete dots)
REL_CMAP   = os.environ.get("REL_CMAP", "inferno")        # any matplotlib cmap (e.g. "viridis", "magma")
REL_SMOOTH = float(os.environ.get("REL_SMOOTH", "0.10"))  # RBF smoothing; ↓ tighter/more detail, ↑ smoother
# ──────────────────────────────────────────────────────────────────────────────
from matplotlib.patches import Ellipse

_REL_LABEL = {"message":  r"$\|\Delta h_{\mathrm{msg}}\|/\|h\|$",
              "reconfig": r"$\|\Delta h_{\mathrm{msg}}\|/\|\Delta h_{\mathrm{self}}\|$",
              "full":     r"$\|\Delta h\|/\|h\|$"}

def _view_xy(pos3d, view):
    p = np.asarray(pos3d, float)
    if view == "axial":     return p[:, [0, 1]]              # top-down: x=right, y=nose(up)
    if view == "sagittal":  return np.c_[-p[:, 1], p[:, 2]]  # from the right: anterior→left, up
    if view == "posterior": return p[:, [0, 2]]              # from behind: x=right, y=up
    raise ValueError(f"unknown view {view!r} (use axial|sagittal|posterior)")

def _interp_field(x, y, vals, cx, cy, rx, ry, smooth, res=180):
    """Smooth scattered interpolation over the head ellipse. RBF thin-plate-spline (clean, MNE-like)
    with a Gaussian-kernel fallback if SciPy is unavailable. `smooth` regularizes: 0 ≈ exact, ↑ smoother."""
    gx = np.linspace(cx - rx, cx + rx, res); gy = np.linspace(cy - ry, cy + ry, res)
    GX, GY = np.meshgrid(gx, gy)
    scale = 0.5 * (rx + ry) + 1e-12
    vlo, vhi = float(np.min(vals)), float(np.max(vals))
    try:
        from scipy.interpolate import RBFInterpolator
        rbf = RBFInterpolator(np.c_[x, y] / scale, vals, kernel="thin_plate_spline",
                              smoothing=max(smooth, 0.0) * len(x))
        F = rbf(np.c_[GX.ravel(), GY.ravel()] / scale).reshape(GX.shape)
    except Exception:                                       # no SciPy → Nadaraya–Watson kernel
        sig = max(smooth, 0.05) * scale
        d2 = (GX[..., None] - x) ** 2 + (GY[..., None] - y) ** 2
        w = np.exp(-d2 / (2.0 * sig ** 2)); F = (w * vals).sum(-1) / np.maximum(w.sum(-1), 1e-12)
    F = np.clip(F, vlo, vhi)                                 # tame TPS overshoot at the rim
    F[((GX - cx) / rx) ** 2 + ((GY - cy) / ry) ** 2 > 1.0] = np.nan
    return F, (gx[0], gx[-1], gy[0], gy[-1])

def draw_view(ax, pos2d, field, title, ei_np=None, cmap="inferno", vmin=None, vmax=None,
              style="smooth", smooth=0.10, decorate="eeg"):
    """One head/brain view. decorate="eeg" adds a nose (axial) / anterior arrow (sagittal);
    decorate="none" draws just the silhouette (for fMRI parcels)."""
    x, y = pos2d[:, 0], pos2d[:, 1]
    cx, cy = (x.max() + x.min()) / 2, (y.max() + y.min()) / 2
    rx, ry = (x.max() - x.min()) / 2 * 1.10, (y.max() - y.min()) / 2 * 1.10
    vmin = float(np.nanmin(field)) if vmin is None else vmin
    vmax = float(np.nanmax(field)) if vmax is None else vmax
    if style == "nodes":
        if ei_np is not None:
            segs = [[(x[s], y[s]), (x[d], y[d])] for s, d in zip(ei_np[0], ei_np[1])]
            ax.add_collection(LineCollection(segs, colors="#cfcfcf", lw=0.5, alpha=0.7, zorder=1))
        mappable = ax.scatter(x, y, s=70, c=field, cmap=cmap, vmin=vmin, vmax=vmax,
                              edgecolors="k", linewidths=0.4, zorder=3)
    else:
        F, ext = _interp_field(x, y, field, cx, cy, rx, ry, smooth)
        mappable = ax.imshow(F, extent=ext, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax,
                             interpolation="bilinear", zorder=2)
        ax.scatter(x, y, s=8, c="k", zorder=5)
    ax.add_patch(Ellipse((cx, cy), 2 * rx, 2 * ry, fill=False, ec="k", lw=1.3, zorder=4))
    if decorate == "eeg" and title == "axial":              # nose at top
        ax.plot([cx - 0.10 * rx, cx, cx + 0.10 * rx], [cy + ry * 0.98, cy + ry * 1.12, cy + ry * 0.98],
                color="k", lw=1.3, zorder=4)
    elif decorate == "eeg" and title == "sagittal":         # anterior marker on the left
        ax.annotate("ant", (cx - rx, cy), (cx - rx * 1.28, cy), fontsize=8, va="center", ha="right",
                    arrowprops=dict(arrowstyle="->", lw=1.0))
    ax.set_title(title, fontsize=11); ax.set_aspect("equal"); ax.axis("off")
    return mappable

def rel_update_field(d, g, mode):
    """(N,) time-averaged relative-update field for a step-decomposition dict `d` and gain `g`."""
    step = float(d.get("step", 1.0)); hn = d["h_norm"] + 1e-12
    rel = {"message":  g * step * d["off_norm"]  / hn,
           "reconfig": d["off_norm"] / (d["diag_norm"] + 1e-12),
           "full":     g * d["update_norm"] / hn}[mode]
    return rel.mean(0)

def rel_update_report(name, d, g, cfg, extra=""):
    step = float(d.get("step", 1.0)); hn = d["h_norm"] + 1e-12
    full = g * d["update_norm"] / hn; diag = g * step * d["diag_norm"] / hn
    off = g * step * d["off_norm"] / hn; reco = d["off_norm"] / (d["diag_norm"] + 1e-12)
    print(f"{name}: sheaf_norm={cfg.get('sheaf_norm','none')}  diffusion_step={cfg.get('diffusion_step',1.0)}  "
          f"gain g={g:.3f}  eff_step={g*step:.3f}{extra}")
    print(f"    median ||Δh||/||h||      (full)     = {np.median(full):6.2f}   <- self-block dominated, ∝ degree×‖ρ‖²")
    print(f"    median ||Δh_self||/||h|| (diag)     = {np.median(diag):6.2f}")
    print(f"    median ||Δh_msg||/||h||  (off)      = {np.median(off):6.2f}   <- neighbour-driven update")
    print(f"    median off/diag          (reconfig) = {np.median(reco):6.2f}   <- gauge-free: >1 graph rewrites node")

views = [v.strip() for v in REL_VIEWS.split(",") if v.strip()] or ["axial"]
for name in EEG_NAMES:
    rec = HARVEST[name]; d = STEP_DECOMP[name]
    gl = rec["model"].dynamics.graph_laplacian
    g  = float(gl.diffusion_gain()) if hasattr(gl, "diffusion_gain") else 1.0
    ei_np = rec["ei"].detach().cpu().numpy()
    field = rel_update_field(d, g, REL_MODE)
    pos3d, _ = montage(rec)
    fig, axes = plt.subplots(1, len(views), figsize=(5.5 * len(views), 5.6), squeeze=False); axes = axes[0]
    vmin, vmax = float(np.nanpercentile(field, 2)), float(np.nanpercentile(field, 98))
    mappable = None
    for ax, view in zip(axes, views):
        mappable = draw_view(ax, _view_xy(pos3d, view), field, view, ei_np=ei_np, cmap=REL_CMAP,
                             vmin=vmin, vmax=vmax, style=REL_STYLE, smooth=REL_SMOOTH, decorate="eeg")
    cbar = fig.colorbar(mappable, ax=list(axes), fraction=0.046, pad=0.04); cbar.set_label(_REL_LABEL[REL_MODE])
    fig.suptitle(f"{name}: relative update per node  ·  {_REL_LABEL[REL_MODE]}  "
                 f"(mode={REL_MODE}, style={REL_STYLE})", fontweight="bold")
    plt.show()
    rel_update_report(name, d, g, rec["ckpt"]["config"],
                      extra=f"  |  views={views} style={REL_STYLE} cmap={REL_CMAP}"
                            + (f" bw={REL_SMOOTH}" if REL_STYLE == "smooth" else ""))
if not EEG_NAMES:
    print("No EEG dataset configured — skipping the relative-update scalp.")

## Figure 3b · Same relative update, on an fMRI checkpoint

Identical decomposition and `REL_*` toggles as Figure 3, but for an **fMRI** run whose nodes are
Schaefer-400 parcels. **Gated** — runs only if a `kind="fmri"` checkpoint was harvested (set
`FMRI_CKPT` in §0 and re-run §1); with EEG only it prints a skip line and needs a `SheafLaplacian`
checkpoint (skips the GCN ablation).

**`FMRI_RENDER`** picks how the volumetric parcels are drawn — a flat axis-drop is a poor fit for a
3-D volume, so the default is a real brain:

- **`glass`** (default) — nilearn glass brain (`plot_markers`): true brain silhouette, native
  plane projection, one marker per parcel coloured by the field. Needs `centroids.json`
  (`scripts/build_schaefer_centroids.py`).
- **`surface`** — nilearn cortical surface (`plot_img_on_surf`): the field painted into the Schaefer
  volume and projected to fsaverage (lateral + medial, both hemispheres) — the closest to the DMN
  reference. Needs the atlas NIfTI via `nilearn.datasets` (cached or online); falls back to `glass`
  if unavailable.
- **`flat`** — the Figure-3 matplotlib topomap on parcel centroids (orthographic axis-drop in a
  bounding oval; honest only because EEG sensors are ~2-D — for a 3-D volume it collapses depth and
  looks lumpy). Honours `REL_VIEWS` / `REL_STYLE`.

`REL_MODE` / `REL_CMAP` / `REL_SMOOTH` carry over; `REL_STYLE` / `REL_VIEWS` apply to `flat` only.
Reuses `rel_update_field`, `rel_update_report`, and `draw_view` from Figure 3.

In [ ]:
# Figure 3b · Relative update per node for an fMRI checkpoint.
# Gated: needs a harvested kind="fmri" SheafLaplacian checkpoint; otherwise it just skips.
FMRI_NAMES  = [k for k, v in DATASETS.items() if v.get("kind") == "fmri" and k in HARVEST]
FMRI_RENDER  = os.environ.get("FMRI_RENDER", "glass")     # "glass" | "surface" | "flat"
FMRI_DISPLAY = os.environ.get("FMRI_DISPLAY", "lzr")      # glass: nilearn display_mode (e.g. lzr, lyrz, ortho)

def _fmri_view_xy(mni, view):
    p = np.asarray(mni, float)                             # MNI: x=L→R, y=post→ant, z=inf→sup
    if view == "axial":     return p[:, [0, 1]]
    if view == "sagittal":  return np.c_[-p[:, 1], p[:, 2]]
    if view == "posterior": return p[:, [0, 2]]
    raise ValueError(view)

def _fmri_centroids(N):
    from data.rbc_dataset import schaefer_centroids
    return schaefer_centroids(expected_nodes=N)

def _render_glass(field, N, cmap, title):
    from nilearn import plotting as niplot
    mni = _fmri_centroids(N)
    vlo, vhi = float(np.nanpercentile(field, 2)), float(np.nanpercentile(field, 98))
    disp = niplot.plot_markers(node_values=np.asarray(field, float), node_coords=mni, node_size=25,
                               node_cmap=cmap, node_vmin=vlo, node_vmax=vhi,
                               display_mode=FMRI_DISPLAY, colorbar=True, title=title)
    plt.show(); disp.close()

def _render_surface(field, N, cmap, title):
    from nilearn import datasets as nid, image as niimg, plotting as niplot
    atl = nid.fetch_atlas_schaefer_2018(n_rois=N, yeo_networks=7, resolution_mm=2)
    aimg = niimg.load_img(atl.maps); adata = np.asarray(aimg.get_fdata())
    stat = np.zeros(adata.shape, np.float32)
    for i, v in enumerate(np.asarray(field, float)):       # parcel i ↔ atlas integer label i+1
        stat[adata == i + 1] = v
    simg = niimg.new_img_like(aimg, stat)
    niplot.plot_img_on_surf(simg, views=["lateral", "medial"], hemispheres=["left", "right"],
                            colorbar=True, cmap=cmap, title=title); plt.show()

def _render_flat(field, N, ei_np, cmap, title):
    mni = _fmri_centroids(N)
    fviews = [v.strip() for v in REL_VIEWS.split(",") if v.strip()] or ["axial"]
    fig, axes = plt.subplots(1, len(fviews), figsize=(5.5 * len(fviews), 5.6), squeeze=False); axes = axes[0]
    vmin, vmax = float(np.nanpercentile(field, 2)), float(np.nanpercentile(field, 98)); m = None
    for ax, view in zip(axes, fviews):
        m = draw_view(ax, _fmri_view_xy(mni, view), field, view, ei_np=ei_np, cmap=cmap,
                      vmin=vmin, vmax=vmax, style=REL_STYLE, smooth=REL_SMOOTH, decorate="none")
    cbar = fig.colorbar(m, ax=list(axes), fraction=0.046, pad=0.04); cbar.set_label(_REL_LABEL[REL_MODE])
    fig.suptitle(title, fontweight="bold"); plt.show()

if not FMRI_NAMES:
    print("No fMRI checkpoint harvested — skipping Figure 3b. "
          "(Set FMRI_CKPT in §0 with kind='fmri' and re-run §1 to enable.)")
else:
    for name in FMRI_NAMES:
        rec = HARVEST[name]; gl = rec["model"].dynamics.graph_laplacian
        if not hasattr(gl, "diffuse"):
            print(f"{name}: graph op is not a SheafLaplacian — skipping."); continue
        try:
            loader, _ = pick_split(rec["loaders"])
            d = rm.sheaf_step_decomposition_over_loader(
                rec["model"], loader, rec["ei"], norm_stats=rec["ckpt"].get("norm_stats"),
                max_batches=MAX_BATCHES, device=device, round_idx=0)
        except TypeError as e:
            print(f"{name}: {e} — skipping."); continue
        g = float(gl.diffusion_gain()) if hasattr(gl, "diffusion_gain") else 1.0
        ei_np = rec["ei"].detach().cpu().numpy()
        field = rel_update_field(d, g, REL_MODE)
        N = rec["N"]
        title = f"{name} (fMRI): relative update  ·  {_REL_LABEL[REL_MODE]}  (mode={REL_MODE}, render={FMRI_RENDER})"
        try:
            if FMRI_RENDER == "surface":
                _render_surface(field, N, REL_CMAP, title)
            elif FMRI_RENDER == "flat":
                _render_flat(field, N, ei_np, REL_CMAP, title)
            else:
                _render_glass(field, N, REL_CMAP, title)
        except (FileNotFoundError, ValueError, ImportError) as e:
            print(f"{name}: {FMRI_RENDER!r} render unavailable ({type(e).__name__}: {e}).")
            try:                                            # graceful fallback: glass, else flat
                _render_glass(field, N, REL_CMAP, title + "  [fallback: glass]")
            except Exception as e2:
                print(f"    glass fallback also failed ({e2}); using flat oval.")
                _render_flat(field, N, ei_np, REL_CMAP, title + "  [fallback: flat]")
        rel_update_report(name, d, g, rec["ckpt"]["config"], extra=f"  |  fMRI / Schaefer-{N}  render={FMRI_RENDER}")

## Figure 3 — dynamic attenuation (gauge-invariant, % update)

Replaces the old scalp `‖ρ‖` figure. Communication per edge is the **diffusion gain as a percentage of
the receiving node's embedding magnitude** (`eff_step·‖ρᵀδ‖ / ‖h‖`) — "how much of an update this edge
delivers" — which is gauge-invariant and needs no graph topology. Three renders: two distribution
histograms, then the edge×time field as either a heatmap (recommended) or a 3D histogram.

In [ ]:
# ── B · publication histograms — spatial + temporal heterogeneity of the % update ───────────
import matplotlib as mpl
_ACCENT, _INK, _CMAP = "#3573b1", "#22303f", "magma"
_PUB = {"figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
        "font.size": 12, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.9,
        "axes.edgecolor": "#4a4a4a", "xtick.color": "#4a4a4a", "ytick.color": "#4a4a4a",
        "axes.labelcolor": _INK}

def _pct_gain(rec):
    # diffusion gain as % of the receiving node's ||h|| (relative mode), per (window, edge).
    dg = harvest_diffusion_gain(rec, "relative")
    return 0.5 * (dg["c_to_src"] + dg["c_to_dst"]) * 100.0            # (S, E) in %

DYN = {name: _pct_gain(HARVEST[name]) for name in EEG_NAMES}          # cache for the plot-3 cells
if not EEG_NAMES:
    print("No EEG dataset — skipping Figure 3 (B).")

with mpl.rc_context(_PUB):
    for name in EEG_NAMES:
        ew = DYN[name]
        mean_e = ew.mean(0)                                          # per-edge mean % update (spatial)
        cov_e = ew.std(0) / (mean_e + 1e-9)                          # temporal variability (CoV)
        fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.9))
        for a, vals, xlabel, is_pct in [
                (ax[0], mean_e, "mean update magnitude  (% of ‖h‖)", True),
                (ax[1], cov_e,  "temporal variability  (CoV)",       False)]:
            hi = float(np.percentile(vals, 99))
            a.hist(vals, bins=np.linspace(0, hi, 31), color=_ACCENT,
                   edgecolor="white", linewidth=0.6, alpha=0.92)
            med = float(np.median(vals))
            a.axvline(med, color=_INK, ls="--", lw=1.2)
            a.annotate(f"median {med:.1f}{'%' if is_pct else ''}",
                       xy=(med, a.get_ylim()[1] * 0.94), xytext=(6, 0),
                       textcoords="offset points", fontsize=10, color=_INK, va="top")
            a.set_xlabel(xlabel); a.set_ylabel("number of edges")
            a.grid(axis="y", color="#e8e8e8", lw=0.8); a.set_axisbelow(True)
            a.margins(x=0.01)
        fig.tight_layout(); plt.show()

In [ ]:
# ── B · plot 3, option 1 (recommended) — edge × time heatmap of the % update ────────────────
# Legible replacement for the overlaid time-courses: every edge is one row, colour = % update. Vertical
# bright bands = moments of high global communication; horizontal bands = persistently-active edges.
with mpl.rc_context(_PUB):
    for name in EEG_NAMES:
        ew = DYN[name]                                              # (S, E) %
        order = np.argsort(-ew.mean(0))                             # strongest edges on top
        M = ew[:, order].T                                         # (E, S)
        vmax = float(np.percentile(M, 99))
        fig, ax = plt.subplots(figsize=(9.6, 4.7))
        im = ax.imshow(M, aspect="auto", cmap=_CMAP, vmin=0, vmax=vmax,
                       extent=[0, M.shape[1], M.shape[0], 0], interpolation="nearest")
        ax.set_xlabel("window (time)"); ax.set_ylabel("edge  (sorted by mean update)")
        cb = fig.colorbar(im, ax=ax, pad=0.015, fraction=0.045)
        cb.set_label("update magnitude  (% of ‖h‖)"); cb.outline.set_visible(False)
        fig.tight_layout(); plt.show()

In [ ]:
# ── B · plot 3, option 2 — 3D histogram: the % update distribution over time ────────────────
# For each time slice, the distribution of per-edge % update; stacked along time into a surface. A burst
# of communication shows as mass shifting toward higher % (a ridge moving right).
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3d projection)
with mpl.rc_context(_PUB):
    for name in EEG_NAMES:
        ew = DYN[name]; S = ew.shape[0]
        n_t = min(28, S)
        tb = np.linspace(0, S, n_t + 1).astype(int)
        gmax = float(np.percentile(ew, 99))
        gb = np.linspace(0, gmax, 30); gc = 0.5 * (gb[:-1] + gb[1:])
        Z, tc = [], []
        for i in range(n_t):
            seg = ew[tb[i]:tb[i + 1]].ravel()
            if seg.size == 0:
                continue
            h, _ = np.histogram(seg, bins=gb, density=True)
            Z.append(h); tc.append(0.5 * (tb[i] + tb[i + 1]))
        Z = np.array(Z)                                            # (n_t, n_gbin) density
        Xg, Yt = np.meshgrid(gc, np.array(tc))
        fig = plt.figure(figsize=(9.6, 6.2))
        ax = fig.add_subplot(projection="3d")
        surf = ax.plot_surface(Xg, Yt, Z, cmap=_CMAP, rstride=1, cstride=1,
                               linewidth=0, antialiased=True)
        ax.set_xlabel("update  (% of ‖h‖)", labelpad=8)
        ax.set_ylabel("window (time)", labelpad=8)
        ax.set_zlabel("edge density", labelpad=6)
        ax.view_init(elev=32, azim=-58)
        try:
            ax.set_box_aspect((1.1, 1.4, 0.7))
        except Exception:
            pass
        cb = fig.colorbar(surf, ax=ax, pad=0.08, fraction=0.03, shrink=0.6)
        cb.outline.set_visible(False)
        fig.tight_layout(); plt.show()

## Figure 4 · Restriction-map response over context (evoked-style)

Inspired by the classic evoked-response layout (topomaps above a butterfly plot), but the panels are
**context windows**, not moments. Bottom: the **actual** continuous EEG for one subject — no prediction.
Above it, one scalp map per context window, coloured by a per-window quantity the model instantiates from
that window's context. Each scalp is tied to the *span* it summarizes (the stride-chunked window), so
left-to-right shows how that quantity **varies over time in response to context**.

**What to colour by — `RESP_METRIC`:** `"rho"` (default) restriction-map norm `‖ρ‖_F` per edge;
`"gain"` the diffusion gain `eff_step·‖msg‖` each edge delivers (gauge-invariant); `"relupdate"` the
per-node relative state change `‖Δh‖/‖h‖`. Edge metrics draw coloured edges, the node metric colours nodes.

**Even spacing.** Snapshot starts are placed on the **continuous block timeline**, not the training window
index — the short EEG arm subsamples starts (`--max_windows_per_subject 150`), so stepping the sampled
windows clumps them. `RESP_SPAN="spread"` (default) spaces windows evenly across the longest block;
`"tile"` places adjacent stride chunks (`RESP_STEP_W`, `RESP_START_W`).

**Absolute vs deviation.** `‖ρ‖` is mostly **structural** (posterior/occipital edges always high) with a
small state-dependent part, so on a shared absolute scale the windows look nearly identical. The cell
prints the **between-site (structural) vs within-site (state) variance split**, and `RESP_CENTER=1`
recolours each scalp by `metric − (that site's temporal mean)` on a diverging scale to isolate the
state-dependent deviation — the view for the "do the norms echo more deviant dynamics?" question. (For
`rho`, raw values also carry a constant `sheaf_norm=sym` gauge scalar that never distorts the *pattern*;
`gain` is gauge-invariant if you'd rather avoid it.)

**Continuous norm strip.** `RESP_NORM_TRACE=1` inserts a thin strip between the scalps and the EEG that
plots the chosen metric averaged over sites as a **continuous line over the same time axis** (dense grid of
`RESP_NORM_TRACE_N` windows, autoscaled to the wiggle); the six panels become dots on that curve, with the
connectors landing on them. This ties the rows into one argument and shows whether the norm rises during
the more turbulent EEG stretches. The strip always shows the absolute metric (even with `RESP_CENTER`).

**Trace toggles** (61 overlaid channels get cluttered): `RESP_CHANNELS` = channel names or 0-based indices
for a subset (`""` = all). `RESP_TRACE` picks the bottom-trace style — `"channels"` (coloured overlay,
default), `"mean"` (one dark mean line, = old `RESP_AVG=1`), or `"grey_mean"` (all channels faint grey + a
bold mean on top, the cleanest for print). The mean does NOT go to zero: EEG channels are spatially
correlated so the shared global component survives averaging. `RESP_UV` (default on) scales the trace
volts→µV (raw LEMON blocks come from `mne.get_data()`, i.e. volts). The maps are always built from the full
channel set; trace toggles only change the bottom curve.

**Total time span (spread mode).** By default `spread` covers the whole block; `RESP_DURATION`
(seconds) shrinks the spanned window and `RESP_START_S` (seconds) sets where it begins (`""` → centred,
clamped to the block). E.g. `RESP_DURATION=5` gives six windows evenly spread across a 5 s stretch.

**Other toggles:** `SFREQ`, `RESP_N_SNAP`, `RESP_SUBJECT` (`""` → longest block), `RESP_CMAP`,
`RESP_BASELINE_N` (windows in the temporal-mean / variance grid).

In [ ]:
# Figure 4 · Restriction-map response over context — per-context-window scalp maps over the actual EEG.
# Top: one scalp per context window, coloured by a chosen per-window quantity (RESP_METRIC), either
# absolute or as a deviation from its temporal mean (RESP_CENTER). Bottom: the real continuous signal for
# one subject (channel subset or a single mean line). Snapshot starts are placed on the CONTINUOUS timeline
# — independent of the training window subsample (--max_windows_per_subject) — so they are evenly spaced.
from matplotlib.patches import ConnectionPatch
from main import batch_to_model_tensors

SFREQ         = float(os.environ.get("SFREQ", "250"))    # LEMON sampling rate (Hz) for the time axis
RESP_METRIC   = os.environ.get("RESP_METRIC", "rho")     # "rho" ||rho||_F (edge) | "gain" diffusion gain (edge) | "relupdate" ||dh||/||h|| (node)
RESP_N_SNAP   = int(os.environ.get("RESP_N_SNAP", "6"))  # number of context-window scalps across the top
RESP_SPAN     = os.environ.get("RESP_SPAN", "spread")    # "spread" = even across the block | "tile" = adjacent stride chunks
RESP_STEP_W   = os.environ.get("RESP_STEP_W", "")        # (tile only) stride-windows advanced between snapshots ("" = x//stride = adjacent)
RESP_START_W  = os.environ.get("RESP_START_W", "")       # (tile only) first snapshot's start in stride-windows ("" = centre)
RESP_SUBJECT  = os.environ.get("RESP_SUBJECT", "")       # subject_id to feature ("" = the longest continuous block)
RESP_CMAP     = os.environ.get("RESP_CMAP", "viridis")
RESP_CHANNELS = os.environ.get("RESP_CHANNELS", "")      # comma-sep channel names or 0-based indices to plot ("" = all)
RESP_AVG      = os.environ.get("RESP_AVG", "0") == "1"   # collapse the selected channels to ONE mean "brain dynamics" line
RESP_CENTER   = os.environ.get("RESP_CENTER", "0") == "1"  # colour scalps by the metric MINUS its temporal mean (reveals state-dependent variation)
RESP_BASELINE_N = int(os.environ.get("RESP_BASELINE_N", "48"))  # dense grid of windows for the temporal mean + variance split
RESP_UV       = os.environ.get("RESP_UV", "1") == "1"    # scale the trace to µV (raw LEMON blocks are volts, from mne get_data)
RESP_DURATION = os.environ.get("RESP_DURATION", "")      # (spread only) seconds of recording to span ("" = whole block)
RESP_START_S  = os.environ.get("RESP_START_S", "")       # (spread only) start of that span in seconds ("" = centre it)
RESP_TRACE    = os.environ.get("RESP_TRACE", "")         # bottom-trace style: ""->from RESP_AVG; "channels" | "mean" | "grey_mean"
RESP_NORM_TRACE   = os.environ.get("RESP_NORM_TRACE", "0") == "1"   # add a continuous norm strip (metric mean over sites) between scalps and EEG
RESP_NORM_TRACE_N = int(os.environ.get("RESP_NORM_TRACE_N", "160")) # dense windows used to draw that strip

_METRIC_LABEL = {"rho": r"restriction-map norm  $\|\rho\|_F$",
                 "gain": r"diffusion gain  eff_step$\cdot\|$msg$\|$",
                 "relupdate": r"relative update  $\|\Delta h\|/\|h\|$"}
_METRIC_KIND  = {"rho": "edge", "gain": "edge", "relupdate": "node"}
_METRIC_SHORT = {"rho": r"$\|\rho\|_F$", "gain": "gain", "relupdate": r"$\|\Delta h\|/\|h\|$"}

def _resolve_channels(channels, spec):
    """Map a comma-sep spec of channel names / 0-based indices to column indices into `channels`."""
    if not spec.strip():
        return list(range(len(channels)))
    lut = {c.lower(): i for i, c in enumerate(channels)}
    idx = []
    for tok in (t.strip() for t in spec.split(",") if t.strip()):
        if tok.lower() in lut:
            idx.append(lut[tok.lower()])
        elif tok.lstrip("+-").isdigit() and 0 <= int(tok) < len(channels):
            idx.append(int(tok))
        else:
            print(f"  [RESP_CHANNELS] '{tok}' is not a channel name or valid index - skipped")
    return idx or list(range(len(channels)))

def _pick_block(ds, subject=""):
    """The LONGEST continuous block (path, subject_id, T) — length is subsample-independent, so the
    bottom trace is a genuine continuous stretch regardless of --max_windows_per_subject."""
    seen = {}
    for path, meta, t in ds._samples:
        seen.setdefault(path, (meta["subject_id"], int(meta.get("T", 0))))
    items = [(p, s, T) for p, (s, T) in seen.items()]
    if subject:
        items = [it for it in items if it[1] == subject] or items
    items.sort(key=lambda it: (it[2], it[0]), reverse=True)
    return items[0]

def _bw_split(val):
    """Structural (between-edge/node) vs state (within, across windows) fraction of a metric's variance."""
    em = val.mean(0); between = float(em.var()); within = float(val.var(0).mean())
    return between / (between + within + 1e-12)

for name in EEG_NAMES:
    rec = HARVEST[name]
    ds, _ = pick_split(rec["loaders"]); ds = ds.dataset
    x, stride = ds.x, ds.stride
    channels = list(ds.channels)
    ch_idx = _resolve_channels(channels, RESP_CHANNELS)
    ei_np = rec["ei"].detach().cpu().numpy()
    gl = rec["model"].dynamics.graph_laplacian
    kind = _METRIC_KIND.get(RESP_METRIC, "edge")
    pos3d, pos2d = montage(rec, "regular")
    sp = spatial_scaffold(pos3d, N_SPATIAL_EDGES) if SHOW_SPATIAL_EDGES else []

    path, subj, _T = _pick_block(ds, RESP_SUBJECT)
    ts = ds._load_cached(path)                            # (T, N) full continuous block, in volts
    T = ts.shape[0]; hi_lim = max(0, T - x)
    n = max(1, min(RESP_N_SNAP, hi_lim + 1))

    if RESP_SPAN == "tile":
        stepw = int(RESP_STEP_W) if RESP_STEP_W else max(1, round(x / stride))
        tile_step = stepw * stride; total = (n - 1) * tile_step
        s0 = int(RESP_START_W) * stride if RESP_START_W else max(0, (hi_lim - total) // 2)
        starts = [min(s0 + k * tile_step, hi_lim) for k in range(n)]
    else:
        if RESP_DURATION:
            dur = max(x, min(int(round(float(RESP_DURATION) * SFREQ)), T))     # samples to span (>=1 window, <= block)
            win_lo = int(round(float(RESP_START_S) * SFREQ)) if RESP_START_S != "" else (T - dur) // 2
            win_lo = max(0, min(win_lo, T - dur)); win_hi = win_lo + dur
        else:
            win_lo, win_hi = 0, T                                             # whole block (default)
        starts = list(np.linspace(win_lo, win_hi - x, n).round().astype(int))

    def _forward_metric(win_starts):
        """Per-window metric on the model's own forward: (m, K) with K=E (edge) or N (node)."""
        ctx = np.stack([ts[t:t + x] for t in win_starts], 0)
        batch = {"x": torch.from_numpy(ctx.copy()).float(),
                 "y": torch.zeros(len(win_starts), ds.y, ts.shape[1])}
        xh, _ = batch_to_model_tensors(batch, device, norm_stats=rec["ckpt"].get("norm_stats"))
        g = float(gl.diffusion_gain()) if hasattr(gl, "diffusion_gain") else 1.0
        with torch.no_grad():
            if RESP_METRIC == "gain":                    # eff_step*||msg|| each edge delivers (both output D^-1/2 halves)
                ec = rm.sheaf_edge_contributions(rec["model"], xh, rec["ei"], round_idx=0)
                cs = (g * ec["c_to_src"]).detach().cpu().numpy()
                cd = (g * ec["c_to_dst"]).detach().cpu().numpy()
                scn = gl._node_scale(rec["ei"], rec["ei"].device, torch.float32).detach().cpu().numpy()
                return 0.5 * (cs * scn[ei_np[0]] + cd * scn[ei_np[1]])
            if RESP_METRIC == "relupdate":               # per-node relative state change ||dh||/||h||
                sd = rm.sheaf_step_decomposition(rec["model"], xh, rec["ei"], round_idx=0)
                return (g * sd["update_norm"] / (sd["h_norm"] + 1e-12)).detach().cpu().numpy()
            Hs = rm.harvest_maps(rec["model"], xh, rec["ei"])   # default: ||rho||_F, mean of the two half-maps
            return 0.5 * (rm.frob(Hs["rho_src"]).reshape(len(win_starts), -1) +
                          rm.frob(Hs["rho_dst"]).reshape(len(win_starts), -1))

    metric_val = _forward_metric(starts)                 # (n, K)

    base_starts = sorted(set(np.linspace(0, hi_lim, max(RESP_BASELINE_N, n)).round().astype(int).tolist()))
    base_val = _forward_metric(base_starts)
    between = _bw_split(base_val)
    if RESP_CENTER:
        disp = metric_val - base_val.mean(0)             # signed deviation from each site's temporal mean
        m = float(np.nanpercentile(np.abs(disp), 99)) or 1.0
        cmap, vlo, vhi = "coolwarm", -m, m
        cbar = _METRIC_LABEL[RESP_METRIC] + " − temporal mean"
    else:
        disp = metric_val; cmap = RESP_CMAP
        vlo, vhi = 0.0, float(np.nanpercentile(metric_val, 99)); cbar = _METRIC_LABEL[RESP_METRIC]

    uv = 1e6 if RESP_UV else 1.0                          # volts -> µV for the displayed trace
    lo, hi = starts[0], starts[-1] + x
    seg = ts[lo:hi][:, ch_idx] * uv; tsec = np.arange(lo, hi) / SFREQ
    ylab_u = "µV" if RESP_UV else "V"

    # bottom-trace style ("" -> derive from RESP_AVG for back-compat)
    style = RESP_TRACE or ("mean" if RESP_AVG else "channels")
    _cdesc = {"mean": f"mean of {seg.shape[1]} channel(s)",
              "grey_mean": f"{seg.shape[1]} channels + mean"}.get(
                  style, f"{seg.shape[1]} channels" if RESP_CHANNELS else "all channels")

    # optional continuous norm strip: the (absolute) metric averaged over sites on a dense grid over the span
    if RESP_NORM_TRACE:
        m_pts = int(min(RESP_NORM_TRACE_N, max(2, starts[-1] - lo + 1)))
        trace_starts = np.unique(np.linspace(lo, starts[-1], m_pts).round().astype(int))
        trace_line = _forward_metric(trace_starts).mean(1)           # mean over edges/nodes, absolute
        trace_t = (trace_starts + x / 2) / SFREQ
        snap_norm = metric_val.mean(1)                               # the shown windows as samples of that curve

    _INK, _ACCENT = "#22303f", "#3573b1"
    fig = plt.figure(figsize=(2.3 * n + 1.6, 7.9 if RESP_NORM_TRACE else 7.2))
    if RESP_NORM_TRACE:
        outer = fig.add_gridspec(2, 1, height_ratios=[1.28, 0.9], hspace=0.30)
        tb = outer[0].subgridspec(2, 1, height_ratios=[1.0, 0.34], hspace=0.40)
        top = tb[0].subgridspec(1, n, wspace=0.15)
        nax = fig.add_subplot(tb[1]); cax = fig.add_subplot(outer[1], sharex=nax)
    else:
        gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 0.95], hspace=0.30)
        top = gs[0].subgridspec(1, n, wspace=0.15)
        cax = fig.add_subplot(gs[1]); nax = None

    # bottom trace
    if style == "mean":
        cax.plot(tsec, seg.mean(1), lw=1.3, color=_INK)
        cax.set_ylabel(f"mean {ylab_u}  ({seg.shape[1]} ch)")
    elif style == "grey_mean":
        cax.plot(tsec, seg, lw=0.4, alpha=0.22, color="#9aa0a6")     # all channels, faint grey
        cax.plot(tsec, seg.mean(1), lw=1.4, color=_INK, zorder=4)    # bold mean on top
        cax.set_ylabel(f"{ylab_u}  ({seg.shape[1]} ch + mean)")
    else:
        cax.plot(tsec, seg, lw=0.6, alpha=(0.6 if seg.shape[1] > 6 else 0.9))
        cax.set_ylabel(ylab_u)
    cax.set_xlim(tsec[0], tsec[-1]); cax.set_xlabel("time (s)"); cax.margins(x=0)
    ytop = cax.get_ylim()[1]

    # continuous norm strip (autoscaled to the wiggle so temporal variation is legible)
    if RESP_NORM_TRACE:
        nax.plot(trace_t, trace_line, color=_ACCENT, lw=1.4, zorder=2)
        nax.scatter((np.asarray(starts) + x / 2) / SFREQ, snap_norm, s=26, color=_INK, zorder=3)
        nax.set_ylabel("mean\n" + _METRIC_SHORT[RESP_METRIC], fontsize=9)
        nax.margins(x=0); nax.set_xlim(tsec[0], tsec[-1]); nax.tick_params(labelbottom=False)

    scalp_axes, art = [], None
    for k, t in enumerate(starts):
        ax = fig.add_subplot(top[k]); scalp_axes.append(ax)
        ttl = f"{(t + x / 2) / SFREQ:.2f} s"
        if kind == "node":
            art = draw_topo_nodes(ax, ei_np, pos2d, disp[k], ttl, cmap=cmap, vmin=vlo, vmax=vhi, spatial=sp)
        else:
            art = draw_topo_edges(ax, ei_np, pos2d, disp[k], ttl, scale01=False,
                                  vmin=vlo, vmax=vhi, cmap=cmap, spatial=sp)
        cax.axvspan(t / SFREQ, (t + x) / SFREQ, color=("0.90" if k % 2 == 0 else "0.82"), alpha=0.7, zorder=0)
        cax.axvline((t + x / 2) / SFREQ, color="0.55", lw=0.7, zorder=1)
        if nax is not None:                                          # connect scalp -> its point on the norm strip
            nax.axvline((t + x / 2) / SFREQ, color="0.85", lw=0.7, zorder=0)
            cxy, ctgt = ((t + x / 2) / SFREQ, snap_norm[k]), nax
        else:
            cxy, ctgt = ((t + x / 2) / SFREQ, ytop), cax
        fig.add_artist(ConnectionPatch(xyA=(0.5, -0.02), coordsA=ax.transAxes,
                                       xyB=cxy, coordsB=ctgt.transData, color="0.55", lw=0.9))
    if art is not None:
        cb = fig.colorbar(art, ax=scalp_axes, fraction=0.02, pad=0.01)
        cb.set_label(cbar); cb.outline.set_visible(False)
    _mode = "deviation from temporal mean" if RESP_CENTER else "absolute"
    fig.suptitle(f"{name}: {RESP_METRIC} response over context — subject {subj}, {n} windows "
                 f"({RESP_SPAN}; x={x}, stride={stride}; trace: {_cdesc}; colour: {_mode})", fontweight="bold")
    plt.show()
    site = "edge" if kind == "edge" else "node"
    print(f"{name}: subject {subj} · block {os.path.basename(path)} (T={T} = {T/SFREQ:.1f}s) · "
          f"{n} windows {RESP_SPAN}, segment {lo/SFREQ:.2f}-{hi/SFREQ:.2f}s  |  metric={RESP_METRIC}  |  trace: {_cdesc}")
    print(f"    {RESP_METRIC} variance split (grid of {len(base_starts)} windows): "
          f"structural/between-{site} {between*100:.1f}%  vs  state/within-{site} {(1-between)*100:.1f}%  "
          f"-> {'RESP_CENTER=1 isolates the state part' if between >= 0.9 else 'state variation is already visible'}")
if not EEG_NAMES:
    print("No EEG dataset configured — skipping the response-over-context figure.")

## Figure 5 · Posterior alpha backbone (the structural part of ‖ρ‖)

The between-edge (structural) component of ‖ρ‖ is small (~15%) but spatially organized. This checks the
obvious biology: in eyes-closed EEG the dominant generators are posterior (occipital/parietal alpha), so
the *persistently* strong restriction maps should concentrate there. Three views: the time-averaged
‖ρ‖_F field on the scalp, its correlation with anterior→posterior electrode position, and a
posterior-vs-other edge contrast.

In [ ]:
# Figure 5 · Posterior alpha backbone — is the persistent (between-edge) part of ||rho|| posterior?
from scipy.stats import spearmanr

for name in EEG_NAMES:
    rec = HARVEST[name]
    R_s, R_d = np.asarray(rec["H"]["rho_src"], float), np.asarray(rec["H"]["rho_dst"], float)  # (S,E,H,De)
    fn = lambda R: rm.frob(R).reshape(R.shape[0], R.shape[1])
    edge_norm = (0.5 * (fn(R_s) + fn(R_d))).mean(0)             # (E,) time-averaged map size per edge
    ei_np = rec["ei"].detach().cpu().numpy()
    pos3d, pos2d = montage(rec, "regular")
    sp = spatial_scaffold(pos3d, N_SPATIAL_EDGES) if SHOW_SPATIAL_EDGES else []
    ap = pos3d[:, 1]                                            # +y = nose (anterior); -y = posterior
    edge_ap = 0.5 * (ap[ei_np[0]] + ap[ei_np[1]])              # edge midpoint on the A-P axis
    rho_ap, p_ap = spearmanr(edge_ap, edge_norm)               # expect NEGATIVE: posterior -> larger norm
    lobes = rm.electrode_groups(list(rec["loaders"]["train"].dataset.channels))["lobe"]
    post = np.isin(lobes, ["occipital", "parietal"])
    edge_post = post[ei_np[0]] & post[ei_np[1]]                # both endpoints posterior

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    lc = draw_topo_edges(axes[0], ei_np, pos2d, edge_norm, f"{name}: mean ‖ρ‖_F", scale01=False,
                         cmap="viridis", spatial=sp)
    fig.colorbar(lc, ax=axes[0], fraction=0.046, label="mean ‖ρ‖_F")
    axes[1].scatter(edge_ap, edge_norm, s=10, alpha=0.5, color="#3573b1")
    axes[1].set_xlabel("edge A–P position  (+y anterior → −y posterior)"); axes[1].set_ylabel("mean ‖ρ‖_F")
    axes[1].set_title(f"norm vs A–P  (Spearman {rho_ap:+.2f}, p={p_ap:.1e})")
    grp = [edge_norm[edge_post], edge_norm[~edge_post]]
    means = [g.mean() if len(g) else np.nan for g in grp]
    sems = [g.std() / np.sqrt(max(1, len(g))) for g in grp]
    axes[2].bar([0, 1], means, yerr=sems, color=["#b2182b", "#94a3b8"], capsize=4)
    axes[2].set_xticks([0, 1])
    axes[2].set_xticklabels([f"post–post\n(n={int(edge_post.sum())})", f"other\n(n={int((~edge_post).sum())})"])
    axes[2].set_ylabel("mean ‖ρ‖_F"); axes[2].set_title("posterior vs other edges")
    fig.suptitle(f"{name}: posterior alpha backbone", fontweight="bold"); fig.tight_layout(); plt.show()
    print(f"{name}: ‖ρ‖ vs A–P Spearman {rho_ap:+.2f} (p={p_ap:.1e}); "
          f"post–post mean {means[0]:.2f} vs other {means[1]:.2f}  "
          f"({'posterior-dominant (matches EC alpha)' if means[0] > means[1] else 'NOT posterior-dominant'})")
if not EEG_NAMES:
    print("No EEG dataset — skipping the posterior backbone figure.")

## Figure 6 · Does the norm trace track alpha, or just broadband amplitude?

The norm strip in Figure 4 co-varies with the EEG, but the trivial explanation is amplitude: bursts →
bigger embeddings → bigger maps. This cell separates the two. Over one subject's block it computes the
per-window mean ‖ρ‖ and, from the continuous signal, an **alpha (8–13 Hz) envelope** (posterior channels,
Hilbert) and a **broadband GFP envelope**, then reports Spearman(norm, alpha), Spearman(norm, broadband),
and the **partial** correlation of norm with alpha *controlling for broadband*. A surviving partial alpha
correlation is the non-trivial result ("map gain reconfigures with alpha, beyond amplitude"); if only the
broadband correlation survives, the honest framing is amplitude-tracking. Uses `RESP_SUBJECT`,
`RESP_NORM_TRACE_N`, `SFREQ`.

In [ ]:
# Figure 6 · alpha vs broadband tracking of the per-window mean ||rho||.
from scipy.signal import butter, filtfilt, hilbert
from scipy.stats import spearmanr, rankdata
from main import batch_to_model_tensors

ALPHA_BAND = (8.0, 13.0)

def _block_pick(ds, subject=""):
    seen = {}
    for path, meta, t in ds._samples:
        seen.setdefault(path, (meta["subject_id"], int(meta.get("T", 0))))
    items = [(p, s, T) for p, (s, T) in seen.items()]
    if subject:
        items = [it for it in items if it[1] == subject] or items
    items.sort(key=lambda it: (it[2], it[0]), reverse=True)
    return items[0]

def _win_norm(rec, ts, starts, x, ds):
    ctx = np.stack([ts[t:t + x] for t in starts], 0)
    batch = {"x": torch.from_numpy(ctx.copy()).float(), "y": torch.zeros(len(starts), ds.y, ts.shape[1])}
    xh, _ = batch_to_model_tensors(batch, device, norm_stats=rec["ckpt"].get("norm_stats"))
    with torch.no_grad():
        Hs = rm.harvest_maps(rec["model"], xh, rec["ei"])
    return (0.5 * (rm.frob(Hs["rho_src"]).reshape(len(starts), -1) +
                   rm.frob(Hs["rho_dst"]).reshape(len(starts), -1))).mean(1)

def _partial_spearman(a, b, c):                                # corr(a, b | c) on ranks
    ra, rb, rc = rankdata(a), rankdata(b), rankdata(c)
    x1 = np.c_[np.ones_like(rc), rc]
    res = lambda y: y - x1 @ np.linalg.lstsq(x1, y, rcond=None)[0]
    return float(np.corrcoef(res(ra), res(rb))[0, 1])

_z = lambda v: (v - v.mean()) / (v.std() + 1e-12)

for name in EEG_NAMES:
    rec = HARVEST[name]; ds, _ = pick_split(rec["loaders"]); ds = ds.dataset
    x = ds.x; SF = float(os.environ.get("SFREQ", "250"))
    path, subj, T = _block_pick(ds, os.environ.get("RESP_SUBJECT", ""))
    ts = ds._load_cached(path)                                 # (T, N) volts
    lobes = rm.electrode_groups(list(ds.channels))["lobe"]
    post = np.isin(lobes, ["occipital", "parietal"])
    b, a = butter(4, [ALPHA_BAND[0] / (SF / 2), ALPHA_BAND[1] / (SF / 2)], btype="band")
    alpha_sig = filtfilt(b, a, ts[:, post] if post.any() else ts, axis=0)
    alpha_env = np.abs(hilbert(alpha_sig, axis=0)).mean(1)     # (T,) posterior alpha envelope
    bb_env = ts.std(1)                                         # (T,) broadband GFP
    m_pts = int(min(int(os.environ.get("RESP_NORM_TRACE_N", "160")), max(2, T - x)))
    starts = np.unique(np.linspace(0, T - x, m_pts).round().astype(int))
    norm_w = _win_norm(rec, ts, starts, x, ds)
    alpha_w = np.array([alpha_env[t:t + x].mean() for t in starts])
    bb_w = np.array([bb_env[t:t + x].mean() for t in starts])
    r_a, p_a = spearmanr(norm_w, alpha_w); r_b, p_b = spearmanr(norm_w, bb_w)
    r_pa = _partial_spearman(norm_w, alpha_w, bb_w)
    tsec = (starts + x / 2) / SF

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.0))
    ax[0].plot(tsec, _z(norm_w), color="#3573b1", lw=1.4, label="mean ‖ρ‖")
    ax[0].plot(tsec, _z(alpha_w), color="#b2182b", lw=1.1, alpha=0.85, label="α env (8–13 Hz)")
    ax[0].plot(tsec, _z(bb_w), color="#6b6b6b", lw=1.0, alpha=0.7, label="broadband GFP")
    ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("z-scored"); ax[0].legend(fontsize=8)
    ax[0].set_title(f"{name}: subject {subj}")
    ax[1].scatter(_z(alpha_w), _z(norm_w), s=14, alpha=0.6, color="#b2182b")
    ax[1].set_xlabel("alpha envelope (z)"); ax[1].set_ylabel("mean ‖ρ‖ (z)")
    ax[1].set_title(f"norm vs alpha  (Spearman {r_a:+.2f})")
    ax[2].bar([0, 1, 2], [r_a, r_b, r_pa],
              color=["#b2182b", "#6b6b6b", "#1a9850"])
    ax[2].axhline(0, color="k", lw=0.8); ax[2].set_xticks([0, 1, 2])
    ax[2].set_xticklabels(["norm~α", "norm~bb", "α | bb\n(partial)"]); ax[2].set_ylabel("Spearman / partial r")
    ax[2].set_title("what the norm tracks")
    fig.tight_layout(); plt.show()
    verdict = ("alpha-specific beyond amplitude" if abs(r_pa) >= 0.2 and abs(r_pa) > 0.5 * abs(r_b)
               else "mostly broadband amplitude" if abs(r_b) > abs(r_a) else "ambiguous")
    print(f"{name}: subject {subj} | norm~alpha {r_a:+.2f} (p={p_a:.1e}), norm~broadband {r_b:+.2f} "
          f"(p={p_b:.1e}), partial alpha|bb {r_pa:+.2f}  ->  {verdict}")
if not EEG_NAMES:
    print("No EEG dataset — skipping the alpha-vs-broadband figure.")

## Figure 7 · Restriction-map alignment matrix (network / region)

For each edge `(u,v)` the two learned maps have an alignment `Alignment(u,v) = ⟨ρ_u, ρ_v⟩_F / (‖ρ_u‖‖ρ_v‖)`
(cosine of the vectorized maps): ~1 = parallel restriction spaces (signal transfer), ~0 = orthogonal
(functional gating). Averaging per network/region pair gives a K×K matrix `M`; high diagonal = intra-network
maps preserve signals within a functional domain, low off-diagonal = the sheaf gates between systems.
Grouping is **electrode lobe** for EEG (frontal/central/temporal/parietal/occipital) and **Yeo-7** for an
fMRI checkpoint (set `SCHAEFER_LABELS_CSV` to a per-node network-label CSV). Alignment is averaged over the
harvested windows.

In [ ]:
# Figure 7 · restriction-map alignment matrix by network/region.
def _node_networks(rec, name):
    kind = DATASETS[name]["kind"]
    chans = list(rec["loaders"]["train"].dataset.channels)
    if kind == "eeg":
        lobes = rm.electrode_groups(chans)["lobe"]
        order = ["frontal", "central", "temporal", "parietal", "occipital"]
        names = [g for g in order if g in set(lobes)] + sorted(set(lobes) - set(order))
        return np.array(lobes), names, "electrode lobe"
    import csv as _csv
    lab_csv = os.environ.get("SCHAEFER_LABELS_CSV", "")
    if lab_csv and os.path.exists(lab_csv):
        rows = [r for r in _csv.reader(open(lab_csv)) if r]
        labs = np.array([(r[1] if len(r) > 1 else r[0]) for r in rows])
        return labs, sorted(set(labs)), "Yeo-7 network"
    return None, None, None

for name in HARVEST:
    rec = HARVEST[name]
    R_s, R_d = np.asarray(rec["H"]["rho_src"], float), np.asarray(rec["H"]["rho_dst"], float)  # (S,E,H,De)
    num = (R_s * R_d).sum((-2, -1))                            # (S,E)  Tr(ρ_srcᵀ ρ_dst)
    den = np.sqrt((R_s ** 2).sum((-2, -1))) * np.sqrt((R_d ** 2).sum((-2, -1)))
    align = (num / (den + 1e-12)).mean(0)                     # (E,) window-averaged edge alignment
    labs, names, label_kind = _node_networks(rec, name)
    if labs is None:
        print(f"{name}: no network labels (fMRI needs SCHAEFER_LABELS_CSV) — skipping"); continue
    ei_np = rec["ei"].detach().cpu().numpy(); idx = {g: i for i, g in enumerate(names)}
    K = len(names); S = np.zeros((K, K)); cnt = np.zeros((K, K))
    for e in range(ei_np.shape[1]):
        ga, gb = idx.get(labs[ei_np[0, e]]), idx.get(labs[ei_np[1, e]])
        if ga is None or gb is None:
            continue
        for i, j in {(ga, gb), (gb, ga)}:                     # symmetric accumulation
            S[i, j] += align[e]; cnt[i, j] += 1
    M = np.where(cnt > 0, S / np.maximum(cnt, 1), np.nan)

    fig, ax = plt.subplots(figsize=(1.05 * K + 2.2, 1.0 * K + 1.6))
    im = ax.imshow(M, cmap="magma", vmin=np.nanmin(M), vmax=np.nanmax(M))
    ax.set_xticks(range(K)); ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticks(range(K)); ax.set_yticklabels(names)
    thr = np.nanmean(M)
    for i in range(K):
        for j in range(K):
            if not np.isnan(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                        color="w" if M[i, j] < thr else "k", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, label=r"mean alignment  $\cos(\rho_u,\rho_v)$")
    diag = float(np.nanmean(np.diag(M))); off = float(np.nanmean(M[~np.eye(K, dtype=bool)]))
    ax.set_title(f"{name}: map alignment by {label_kind}\nintra {diag:.2f}  vs  inter {off:.2f}")
    fig.tight_layout(); plt.show()
    print(f"{name}: alignment by {label_kind} | intra-group {diag:.2f} vs inter-group {off:.2f}  "
          f"({'intra > inter: within-domain transfer, between-domain gating' if diag > off else 'no intra>inter gating pattern'})")

## Figure 8 — Qualitative forecast fit (best individual traces)

A **candidate is one node's forecast in one window** — the pool is `W windows × N nodes` traces (e.g. 512 × 400 ≈ 205k). Every candidate is run through the model (its *trained* forecast — short→direct, long→AR), scored by MSE (or normalised DTW) over its forecast steps, and **all candidates are ranked together**. The figure draws the lowest-error traces — each subplot is a different `(window, node)` trace, labelled with its window index, node index, MSE and DTW.

- `QUAL_SELECT` — `best_mse` / `best_dtw` / `random`;
- `QUAL_RANK_K` — start rank (`1` = the single lowest-error trace; `50` starts at the 50th);
- `QUAL_NODES` — how many traces to draw;
- `QUAL_MIN_STD` — optional floor on truth std to skip trivially-flat traces (0 = off).

Other knobs: `QUAL_DATASET`, `QUAL_BATCHES`, `QUAL_HORIZON` and `QUAL_LONG_CHUNK` (long only — the AR chunk defaults to the trained horizon `y`, matching test-time rollout).

In [ ]:
# Figure 8 - Qualitative forecast fit.
# A CANDIDATE is ONE node's forecast in ONE window.  Pool = W windows x N nodes.
# ALGORITHM: score EVERY candidate trace by MSE (or normalised DTW) over its H
# forecast steps, rank ALL of them, and draw the lowest-error traces (starting at
# rank QUAL_RANK_K). Each subplot is a different (window, node) trace = the actual
# best fits, not arbitrary nodes of an "average" window.
import copy
from main import batch_to_model_tensors, rollout_autoregressive
from model.braindyn import BrainDyn, BrainDynConfig

QUAL_DATASET = os.environ.get("QUAL_DATASET", next(iter(HARVEST)))
SELECT       = os.environ.get("QUAL_SELECT", "best_mse")   # random | best_mse | best_dtw
QUAL_RANK_K  = int(os.environ.get("QUAL_RANK_K", "1"))     # start rank (1 = single lowest)
QUAL_NODES   = int(os.environ.get("QUAL_NODES", "6"))      # how many traces to draw
QUAL_BATCHES = int(os.environ.get("QUAL_BATCHES", "8"))    # window-pool batches
QUAL_SEED    = int(os.environ.get("QUAL_SEED", "0"))
QUAL_MIN_STD = float(os.environ.get("QUAL_MIN_STD", "0"))  # drop near-flat truth traces (0=off)

Hd = HARVEST[QUAL_DATASET]
ei, dt = Hd["ei"], Hd["dt"]
ckpt = Hd["ckpt"]; cfg = dict(ckpt["config"]); norm_stats = ckpt.get("norm_stats")

def _build_forecast_model(ckpt, ei, device):
    """Full-fidelity rebuild for FORECASTING. rm.build_model_from_ckpt / config_from_ckpt
    drop forward-only fields (notably sheaf_norm), so the ODE integrates with the WRONG
    operator and predictions blow up to ~15 sigma even though every weight loads. This
    threads every architecture + forward field main.py sets, then loads the FULL state
    dict and hard-fails on any missing/unexpected key."""
    c = dict(ckpt["config"]); g = c.get
    Nn = rm.num_nodes_from_ckpt(ckpt)
    sd = ckpt["model_state_dict"]
    conf = BrainDynConfig(
        signal_dim=1, hidden_dim=g("hidden_dim"), num_nodes=Nn, window_size=g("x"),
        lstm_layers=g("lstm_layers", 1), lstm_dropout=g("lstm_dropout", 0.0),
        map_hidden_dim=g("map_hidden_dim", 16), vf_hidden_dim=g("vf_hidden_dim", 128),
        vf_layers=g("vf_layers", 2),
        use_gcn=bool(g("ablation_gcn") or g("ablation_gat") or g("ablation_simple_graph")),
        use_lstm_encoder=not g("ablation_no_lstm", False),
        edge_specific_maps=g("static_edge_maps", False), sheaf_layers=g("sheaf_layers", 1),
        diffusion_step=g("diffusion_step", 1.0), sheaf_mlp_maps=not g("static_edge_maps", False),
        map_mlp_hidden_dim=g("map_mlp_hidden_dim", 64),
        identity_restriction_init=g("identity_restriction_init", False),
        sheaf_map_pe=g("sheaf_map_pe", "none"), sheaf_map_pe_dim=g("sheaf_map_pe_dim", 8),
        frozen_identity=g("no_sheaf", False), sheaf_norm=g("sheaf_norm", "none"),
        learn_diffusion_gain=g("learn_diffusion_gain", False),
        sheaf_map_scale=g("sheaf_map_scale", "none"), freeze_map_scale=g("freeze_map_scale", False),
        coupling_block=g("coupling_block", "none"),
        time_embed_dim=g("time_embed_dim", 16), time_embed_max_period=g("time_embed_max_period", 16.0),
        time_rate_mode=g("time_rate_mode", "fixed"), time_learn_freqs=g("time_learn_freqs", False),
        time_max_cycles_per_step=g("time_max_cycles_per_step", 0.5))
    sheaf_node_pe = None
    if g("sheaf_map_pe", "none") == "lappe":
        from model.graph_builders import build_lap_pe
        w = sd.get("dynamics.graph_laplacian.map_pe")
        rank = int(w.shape[1]) if w is not None else g("sheaf_map_pe_dim", 8)
        sheaf_node_pe = build_lap_pe(ei.detach().cpu(), num_nodes=Nn, rank=rank)
    m = BrainDyn(conf, sheaf_node_pe=sheaf_node_pe); m.register_restriction_edges(ei)
    miss, unexp = m.load_state_dict(sd, strict=False)
    if list(miss) or list(unexp):
        raise RuntimeError(
            f"forecast model load incomplete: {len(miss)} missing, {len(unexp)} unexpected "
            f"(missing={list(miss)[:4]}, unexpected={list(unexp)[:4]}). A config field is being dropped.")
    return m.to(device).eval()

model = _build_forecast_model(ckpt, ei, device)
print(f"forecast model rebuilt @ full fidelity | sheaf_norm={cfg.get('sheaf_norm','none')!r} "
      f"coupling_block={cfg.get('coupling_block','none')!r} vf_layers={cfg.get('vf_layers',2)} "
      f"learn_diffusion_gain={cfg.get('learn_diffusion_gain',False)}")

# forecast the checkpoint was actually trained to make (auto-detected)
fmode = str(cfg.get("forecast_mode", "short")); is_long = fmode in ("long", "long_ar_train")
if is_long:
    HORIZON  = int(os.environ.get("QUAL_HORIZON", str(cfg.get("test_rollout_steps") or 60)))
    # Faithful to run_test_rollout_chunks: it rolls out in chunk_steps=y (the trained
    # horizon), NOT ar_chunk_size. Stepping 1-at-a-time is off-distribution and jagged.
    AR_CHUNK = int(os.environ.get("QUAL_LONG_CHUNK", str(cfg.get("y", 10))))
    kind_label = f"long AR ({HORIZON} steps, {AR_CHUNK}-step chunks)"
else:
    HORIZON  = int(cfg.get("y", 10)); kind_label = f"short (direct, {HORIZON} steps)"
print(f"{QUAL_DATASET}: forecast_mode={fmode!r} -> {kind_label}")

lc = copy.deepcopy(cfg); lc["y"] = HORIZON; lc["stride"] = max(int(cfg.get("stride", 10)), HORIZON)
q_loader, q_split = pick_split(build_loaders(DATASETS[QUAL_DATASET], lc))

# -- run the model over the pool; keep every window's full node set ------------
model.eval(); CTX, TRUE, PRED = [], [], []
with torch.no_grad():
    for bi, batch in enumerate(q_loader):
        if bi >= QUAL_BATCHES:
            break
        x_hist, y_true = batch_to_model_tensors(batch, ei.device, norm_stats=norm_stats)
        if y_true.shape[0] < HORIZON:
            continue
        if is_long:
            y_pred = rollout_autoregressive(model=model, x_history=x_hist, edge_index=ei,
                                            dt=dt, pred_steps=HORIZON, chunk_size=AR_CHUNK)
        else:
            y_pred = model(x_history=x_hist, edge_index=ei, pred_steps=HORIZON,
                           dt=dt, autoregressive=False)["x_pred"]
        CTX.append(x_hist[..., 0].permute(0, 2, 1).cpu().numpy())            # (B, Lx, N)
        TRUE.append(y_true[:HORIZON, :, :, 0].permute(1, 0, 2).cpu().numpy())  # (B, H, N)
        PRED.append(y_pred[:HORIZON, :, :, 0].permute(1, 0, 2).cpu().numpy())  # (B, H, N)
CTX = np.concatenate(CTX); TRUE = np.concatenate(TRUE); PRED = np.concatenate(PRED)
W, H, N = TRUE.shape; Lx = CTX.shape[1]
assert W > 0, "no windows pooled"

# -- flatten to ONE candidate per (window, node) -------------------------------
ctx = CTX.transpose(0, 2, 1).reshape(W * N, Lx)   # (P, Lx)
tru = TRUE.transpose(0, 2, 1).reshape(W * N, H)   # (P, H)
prd = PRED.transpose(0, 2, 1).reshape(W * N, H)   # (P, H)
P = W * N
win_id  = np.repeat(np.arange(W), N)
node_id = np.tile(np.arange(N), W)
mse = ((prd - tru) ** 2).mean(axis=1)             # (P,) per-trace MSE

def dtw_per_trace(p, t):                           # (M,H) -> (M,) normalised DTW, chunked
    Hh = p.shape[1]; block = max(1000, 20_000_000 // (Hh * Hh))
    out = np.empty(p.shape[0])
    for s in range(0, p.shape[0], block):
        pp = p[s:s + block].astype(np.float64); tt = t[s:s + block].astype(np.float64)
        M = pp.shape[0]
        cost = np.abs(pp[:, :, None] - tt[:, None, :])          # (M,H,H)
        dp = np.full((M, Hh, Hh), np.inf); dp[:, 0, 0] = cost[:, 0, 0]
        for r in range(1, Hh): dp[:, r, 0] = dp[:, r - 1, 0] + cost[:, r, 0]
        for c in range(1, Hh): dp[:, 0, c] = dp[:, 0, c - 1] + cost[:, 0, c]
        for r in range(1, Hh):
            for c in range(1, Hh):
                dp[:, r, c] = cost[:, r, c] + np.minimum(
                    np.minimum(dp[:, r - 1, c], dp[:, r, c - 1]), dp[:, r - 1, c - 1])
        out[s:s + block] = dp[:, Hh - 1, Hh - 1] / max(2 * Hh - 1, 1)
    return out

# -- rank ALL candidates, pick the traces to draw ------------------------------
valid = np.ones(P, bool)
if QUAL_MIN_STD > 0:
    valid &= tru.std(axis=1) >= QUAL_MIN_STD       # skip trivially-flat truth traces
idx_all = np.flatnonzero(valid)
assert idx_all.size, "no candidate traces left after QUAL_MIN_STD filter"

if SELECT == "random":
    chosen = np.random.default_rng(QUAL_SEED).choice(
        idx_all, size=min(QUAL_NODES, idx_all.size), replace=False)
    tag = "random"
elif SELECT in ("best_mse", "best_dtw"):
    metric = mse[idx_all] if SELECT == "best_mse" else dtw_per_trace(prd[idx_all], tru[idx_all])
    order = idx_all[np.argsort(metric)]            # ascending -> lowest error first
    start = int(np.clip(QUAL_RANK_K, 1, order.size)) - 1
    chosen = order[start:start + QUAL_NODES]
    tag = f"lowest {SELECT.split('_')[1].upper()}" + (f" from rank {start + 1}" if start else "")
else:
    raise ValueError(f"QUAL_SELECT must be random|best_mse|best_dtw, got {SELECT!r}")

dtw_show = dtw_per_trace(prd[chosen], tru[chosen])  # DTW labels for the drawn traces

# -- plot: each subplot is ONE ranked (window, node) trace, with its context ----
t_ctx = np.arange(-Lx, 0); t_f = np.arange(H)
C_CTX, C_TRUE, C_PRED = "#7f7f7f", "#1f77b4", "#d62728"
nshow = len(chosen)
fig, axes = plt.subplots(nshow, 1, figsize=(8.5, 1.7 * nshow), sharex=True, squeeze=False)
for r, ci in enumerate(chosen):
    ax = axes[r, 0]
    ax.axvspan(-Lx, -0.5, color=C_CTX, alpha=0.08, lw=0)
    ax.plot(t_ctx, ctx[ci], color=C_CTX, lw=1.5, label="context")
    ax.plot(t_f, tru[ci], color=C_TRUE, lw=2.0, label="truth")
    ax.plot(t_f, prd[ci], color=C_PRED, lw=1.8, ls="--", label="prediction")
    ax.axvline(-0.5, color="k", lw=0.8, alpha=0.5); ax.grid(alpha=0.25)
    ax.set_ylabel(f"win {int(win_id[ci])}\nnode {int(node_id[ci])}", fontsize=8)
    ax.set_title(f"MSE {mse[ci]:.3g}  ·  DTW {dtw_show[r]:.3g}", fontsize=9, loc="right")
    if r == 0:
        ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
axes[-1, 0].set_xlabel("time step (0 = first forecast)")
fig.suptitle(f"{QUAL_DATASET} - {kind_label} - {tag} of {P:,} (window,node) traces  · split={q_split}",
             fontsize=12, y=1.0)
fig.tight_layout(rect=(0, 0, 1, 0.98)); plt.show()
print(f"pool={P:,} traces ({W} windows x {N} nodes) | showing {tag} | (win,node,MSE): " +
      ", ".join(f"({int(win_id[c])},{int(node_id[c])},{mse[c]:.3g})" for c in chosen))
